In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

train_df = pd.read_csv("UNSW_NB15_training-set.csv")
test_df = pd.read_csv("UNSW_NB15_testing-set.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (175341, 45)
Test shape: (82332, 45)


In [ ]:
train_df['label'].value_counts()

,count
label,
1,119341
0,56000


In [ ]:
train_df['label'].value_counts(normalize=True) * 100

,proportion
label,
1,68.062233
0,31.937767


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 45 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 175341 non-null  int64  
 1   dur                175341 non-null  float64
 2   proto              175341 non-null  object 
 3   service            175341 non-null  object 
 4   state              175341 non-null  object 
 5   spkts              175341 non-null  int64  
 6   dpkts              175341 non-null  int64  
 7   sbytes             175341 non-null  int64  
 8   dbytes             175341 non-null  int64  
 9   rate               175341 non-null  float64
 10  sttl               175341 non-null  int64  
 11  dttl               175341 non-null  int64  
 12  sload              175341 non-null  float64
 13  dload              175341 non-null  float64
 14  sloss              175341 non-null  int64  
 15  dloss              175341 non-null  int64  
 16  si

In [ ]:
train_df.isnull().sum().sum()

np.int64(0)

In [ ]:
train_df.columns

Index(['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes',
       'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss',
       'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin',
       'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth',
       'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm',
       'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm',
       'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label'],
      dtype='object')

In [ ]:
# Drop unnecessary columns
train_df = train_df.drop(['id', 'attack_cat'], axis=1)
test_df = test_df.drop(['id', 'attack_cat'], axis=1)

print("New shape:", train_df.shape)

New shape: (175341, 43)


In [ ]:
X_train = train_df.drop("label", axis=1)
y_train = train_df["label"]

X_test = test_df.drop("label", axis=1)
y_test = test_df["label"]

In [ ]:
categorical_cols = X_train.select_dtypes(include=['object']).columns
categorical_cols

Index(['proto', 'service', 'state'], dtype='object')

In [ ]:
X_train = pd.get_dummies(X_train, columns=categorical_cols)
X_test = pd.get_dummies(X_test, columns=categorical_cols)

# Align train and test columns
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print("Final feature count:", X_train.shape[1])

Final feature count: 194


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.73      0.84     37000
           1       0.82      0.99      0.89     45332

    accuracy                           0.87     82332
   macro avg       0.90      0.86      0.86     82332
weighted avg       0.89      0.87      0.87     82332



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

log_pred = log_model.predict(X_test_scaled)

print("Logistic Regression Results:")
print(classification_report(y_test, log_pred))

Logistic Regression Results:
              precision    recall  f1-score   support

           0       0.95      0.61      0.75     37000
           1       0.75      0.98      0.85     45332

    accuracy                           0.81     82332
   macro avg       0.85      0.79      0.80     82332
weighted avg       0.84      0.81      0.80     82332



In [ ]:
from sklearn.svm import SVC

svm_model = SVC()
svm_model.fit(X_train_scaled, y_train)

svm_pred = svm_model.predict(X_test_scaled)

print("SVM Results:")
print(classification_report(y_test, svm_pred))

SVM Results:
              precision    recall  f1-score   support

           0       0.99      0.59      0.74     37000
           1       0.75      0.99      0.86     45332

    accuracy                           0.82     82332
   macro avg       0.87      0.79      0.80     82332
weighted avg       0.86      0.82      0.80     82332



In [ ]:
pip install joblib

In [ ]:
best_model = model

In [ ]:
import joblib

# Save the best model (RandomForestClassifier) to a joblib file
joblib.dump(best_model, 'random_forest_model.joblib')

print("Random Forest model saved as 'random_forest_model.joblib'")

Random Forest model saved as 'random_forest_model.joblib'


In [ ]:
import joblib

# Option 1: Save the scaler separately
joblib.dump(scaler, 'scaler.joblib')
print("Scaler saved as 'scaler.joblib'")

# Option 2: (Recommended) Create and save a pipeline
from sklearn.pipeline import Pipeline

# Define the pipeline steps
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('random_forest', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train the pipeline (this will fit the scaler and then the model)
pipeline.fit(X_train, y_train)

# Save the pipeline
joblib.dump(pipeline, 'random_forest_pipeline.joblib')
print("Random Forest pipeline saved as 'random_forest_pipeline.joblib'")


Scaler saved as 'scaler.joblib'
Random Forest pipeline saved as 'random_forest_pipeline.joblib'


In [ ]:
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# 1. Reload your data (or use the existing train_df)
# Ensure we have the raw features (X) and labels (y)
# We use 'drop' to get the raw features including the text columns
X_train_raw = train_df.drop("label", axis=1)
y_train = train_df["label"]

# 2. Define which columns are categorical and which are numeric
# (Based on your notebook, these are the text columns)
categorical_cols = ['proto', 'service', 'state']
numeric_cols = X_train_raw.select_dtypes(include=['int64', 'float64']).columns.difference(categorical_cols)

# 3. Create the Preprocessor
# This handles converting text to numbers AND scaling numbers automatically
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# 4. Create the Complete Pipeline
# Step 1: Preprocessor (Encode text + Scale numbers)
# Step 2: Classifier (Random Forest)
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# 5. Train the pipeline on the RAW data
print("Training complete pipeline...")
full_pipeline.fit(X_train_raw, y_train)

# 6. Save the CORRECT joblib file
joblib.dump(full_pipeline, 'iot_attack_detection_pipeline.joblib')
print("Saved 'iot_attack_detection_pipeline.joblib'")

# Optional: Verify it works on the test set
X_test_raw = test_df.drop("label", axis=1)
y_test = test_df["label"]
accuracy = full_pipeline.score(X_test_raw, y_test)
print(f"Pipeline Accuracy: {accuracy:.4f}")

Training complete pipeline...
Saved 'iot_attack_detection_pipeline.joblib'
Pipeline Accuracy: 0.8706


In [ ]:
# Install fastapi, uvicorn (server), and pyngrok (to make it accessible)
!pip install fastapi uvicorn pyngrok nest_asyncio python-multipart

In [ ]:
# Install API libraries and localtunnel
!pip install fastapi uvicorn python-multipart pydantic
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 22 packages in 5s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [ ]:
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import joblib
import subprocess
import time
import os
import threading
import urllib.request

# --- 1. DEFINE THE API ---

app = FastAPI(title="IoT Anomaly Detection API")

# Load the model
model_path = "/content/iot_attack_detection_pipeline.joblib"
if not os.path.exists(model_path):
    print("ERROR: Model file not found! Please ensure 'iot_attack_detection_pipeline.joblib' is in the files tab.")
    model = None
else:
    model = joblib.load(model_path)
    print("✅ Model loaded successfully.")

# Define Input Schema (All 43 features with defaults)
class TrafficData(BaseModel):
    dur: float = 0.0
    proto: str = "tcp"
    service: str = "-"
    state: str = "FIN"
    spkts: int = 0
    dpkts: int = 0
    sbytes: int = 0
    dbytes: int = 0
    rate: float = 0.0
    sttl: int = 0
    dttl: int = 0
    sload: float = 0.0
    dload: float = 0.0
    sloss: int = 0
    dloss: int = 0
    sinpkt: float = 0.0
    dinpkt: float = 0.0
    sjit: float = 0.0
    djit: float = 0.0
    swin: int = 0
    stcpb: int = 0
    dtcpb: int = 0
    dwin: int = 0
    tcprtt: float = 0.0
    synack: float = 0.0
    ackdat: float = 0.0
    smean: int = 0
    dmean: int = 0
    trans_depth: int = 0
    response_body_len: int = 0
    ct_srv_src: int = 0
    ct_state_ttl: int = 0
    ct_dst_ltm: int = 0
    ct_src_dport_ltm: int = 0
    ct_dst_sport_ltm: int = 0
    ct_dst_src_ltm: int = 0
    is_ftp_login: int = 0
    ct_ftp_cmd: int = 0
    ct_flw_http_mthd: int = 0
    ct_src_ltm: int = 0
    ct_srv_dst: int = 0
    is_sm_ips_ports: int = 0

@app.get("/")
def home():
    return {"status": "API is running"}

@app.post("/predict")
def predict(traffic: TrafficData):
    if model is None:
        raise HTTPException(status_code=500, detail="Model not loaded")
    try:
        input_df = pd.DataFrame([traffic.model_dump()])
        prediction = model.predict(input_df)
        probability = model.predict_proba(input_df)

        result = "Attack" if prediction[0] == 1 else "Normal"
        confidence = float(probability[0][prediction[0]])

        return {
            "prediction": result,
            "confidence": round(confidence * 100, 2),
            "input_received": traffic.model_dump()
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# --- 2. START SERVER AND TUNNEL ---

# Function to run Uvicorn in a background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Start the server thread
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2) # Wait for server to initialize

# Get Public IP (Needed for Localtunnel Password)
try:
    public_ip = urllib.request.urlopen('https://v4.ident.me').read().decode('utf8')
    print(f"🌍 Your Public IP (Password for Localtunnel): {public_ip}")
except:
    print("Could not fetch public IP. You might need to find it manually.")

# Start Localtunnel
# We use subprocess.PIPE to capture the output and find the URL
print("🚀 Starting Tunnel...")
tunnel_process = subprocess.Popen(['lt', '--port', '8000'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Read the URL from the process output
time.sleep(3) # Give localtunnel time to connect
output = tunnel_process.stdout.readline().decode('utf-8').strip()

if "your url is" in output:
    tunnel_url = output.split("your url is: ")[-1]
    print("\n" + "="*60)
    print(f"🔗 ACCESS YOUR API HERE: {tunnel_url}/docs")
    print("="*60)
    print("\n1. Click the link above.")
    print("2. If asked for a password, enter your Public IP shown above.")
    print("3. You can close the connection by stopping this cell.")
else:
    print("Error creating tunnel. Output:")
    print(output)
    # Sometimes output comes later, print raw lines
    for i in range(5):
        line = tunnel_process.stdout.readline().decode('utf-8').strip()
        if line: print(line)

# Keep the cell alive so the tunnel doesn't close
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Stopping tunnel...")
    tunnel_process.terminate()

INFO:     Started server process [1580]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ Model loaded successfully.
🌍 Your Public IP (Password for Localtunnel): 35.198.241.197
🚀 Starting Tunnel...

🔗 ACCESS YOUR API HERE: https://new-boxes-go.loca.lt/docs

1. Click the link above.
2. If asked for a password, enter your Public IP shown above.
3. You can close the connection by stopping this cell.
INFO:     37.111.211.130:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     37.111.211.130:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     37.111.211.130:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     37.111.211.130:0 - "POST /predict HTTP/1.1" 200 OK
